In [1]:
from dotenv import load_dotenv
load_dotenv()

True

创建智能体

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os

# deepseek
agent = create_agent(model="deepseek-chat")

# 国内其他模型
model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",  # 国内模型模仿openai
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
    api_key = os.getenv("DASHSCOPE_API_KEY")
)

模型调用

In [ ]:
# invoke 阻塞式调用
response = agent.invoke({
    "messages": [{"role": "user", "content": "......."}]
})

print(response)

In [ ]:
# stream 流式调用
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "......"}]},
    stream_mode="messages"
):
    if token.content:
        print(token.content, end="", flush=True)

消息

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-chat",
)

response = agent.invoke({
    "messages":[
        HumanMessage(content=""),
        AIMessage(content=""),
        HumanMessage(content="")
    ]
})

for message in response["messages"]:
    message.pretty_print()



"""
多模态消息
模型要支持多模态
"""
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model = "qwen3.7-plus",
    model_provider="openai",
    base_url = os.getenv("modelurl"),
    api_key = os.getenv("apikey")
)

agent = create_agent(model=model)

# 多模态消息
# 在线图片
multimodal_message = HumanMessage(
    content=[
        {"type": "image", "url": "pit_url"},
        {"type": "text", "text": "......."}
    ]
)

for token, metadata in agent.stream(
    {"messages": [multimodal_message]},
    stream_mode="messages"
):
    if token.content:
        print(token.content, end="", flush=True)






# 本地图片
# 需要先将图片数据转换为base64字符串形式
import base64

def encode_image(image_path: str) -> tuple[str, str]:
    ext = image_path.split('.')[-1].lower()

    mime_map = {
        'jpg': 'image/jpeg', 'jepg': 'image/jpeg',
        'png': 'image/png', 'gif': 'image/gif',
        'webp': 'image/webp'
    } 
    mime_type = mime_map.get(ext, 'image/jpeg')

    with open(image_path, "rb") as f:
        base64_str = base64.b64encode(f.read()).decode("utf-9")

    return base64_str, mime_type

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    base_url = "",
    api_key = ""
)

b64_str, mime = encode_image("image_url")

multimodal_message = HumanMessage(content=[
    {
        "type": "image",
        "base64": b64_str,
        "mime_type": mime
    },
    {"type": "text", "text": ""}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_message]},
    stream_mode="messages"
):
    if chunk.content:
        print(chunk.content, end="", flush=True)

提示词工程

In [ ]:
system_prompt = """
# 身份
- 
- 
.......  描述AI的职责、沟通风格和总体目标
# 说明
-
- 
.......  模型遵守的规则，哪些应该做什么，哪些不能做
# 示例
- 
- 
.......  提供输入示例，以及期望的输出示例
# 背景信息
- 
- 
.......  向模型提供生成响应所需的任何额外信息
"""

agent = create_agent(
    model="",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="")]}
):
    if token.content:
        print(token.content, end="", flush=True)

结构化输出

In [ ]:
from pydantic import BaseModel
class Info(BaseModel):
    name: str
    location: str

agent = create_agent(
    model = "",
    system_prompt= system_prompt,
    response_format= Info
)

response = agent.invoke(
    {"messages": [HumanMessage(content="")]}
)

工具

In [ ]:
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

# 工具：
# 工具名：函数名
# 工具参数：函数参数
# 工具作用：函数的注释
@tool
def square_root(x: float) -> float:
    """Calculate the square root if a number"""
    return x ** 0.5

# 如果参数信息复杂，就要定义一个参数模型来添加参数信息
class WeatherInput(BaseModel):
    location: str = Field(description="")
    units: Literal["a", "b"] = Field(description="")
    forecast: bool = Field(
        default=False,
        description=""
    )

@tool(args_schema=WeatherInput)
def get_weather(location:str, units:str="celsius", forecast:bool=False) -> str:
    result = f"weather"
    return result

# 调用工具
# 如同普通函数的调用
square_root.invoke({"x": 65})

get_weather.invoke({"location": "foshan", "forecast":True})


# 将工具传递给智能体
from langchain.agents import create_agent

agent = create_agent(
    model="deepseek-chat",
    tools=[square_root, get_weather],
    system_prompt=system_prompt
)

# 智能体会自动根据用户问题判断
# 是否调用工具
# 调用哪些工具
# 传递哪些参数

记忆

In [ ]:
# InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver

# 创建智能体
agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver()
)

from langchain.messages import HumanMessage

# thread_id 作为会话标识
config = {"configurable": {"thread_id": "thread_1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content=".....")]},
    config
)

response = agent.invoke(
    {"messages": [HumanMessage(content=".....")]},
    config
)

持久化Memory

In [ ]:
# sqlite
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 初始化checkpointer
checkpointer = SqliteSaver(sqlite3.connect("checkpoint.db", check_same_thread=False))

# 建表
checkpointer.setup()

agent = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer
)



- 修剪消息：在AgentState的消息列表依然完整，但在发给大模型前只保留一部分消息
- 删除消息：删除部分在AgentState中保存的消息
- 总结消息：把历史的消息利用大模型总结出摘要，然后把最新的消息拼接在一起作为新的消息列表发送给大模型

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain.messages import HumanMessage

checkpointer = InMemorySaver()

middleware = SummarizationMiddleware(
    model="deepseek-chat",
    # 触发器
    trigger=("messages", 3),  # 消息数 token数  fraction上下文大小比例
    # 保留消息
    keep=("messages", 1)  # 消息数 token数  fraction上下文大小比例 
)

agent = create_agent(
    model = "deepseek-chat",
    middleware=[middleware],    
    checkpointer=checkpointer
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage(content="")]}, config)
agent.invoke({"messages": [HumanMessage(content="")]}, config)
agent.invoke({"messages": [HumanMessage(content="")]}, config)

response = agent.invoke({"messages": "......"}, config)

for message in response["messages"]:
    message.pretty_print()